# Kaggle: predict + submit

Attach as Datasets: the competition data, the `cell-tracking-src` zip from `scripts/package_for_kaggle.py` (code + `detector.pt` + `ARTIFACT_MANIFEST.json`), and `cell-tracking-wheels` (built once by `notebooks/kaggle_build_wheels.ipynb` with internet on, only needed if `blosc2`/`zarr`/`zstandard` aren't already in the base image). This notebook itself runs with internet OFF, since it's what gets scored.

In [ ]:
import subprocess, sys
from pathlib import Path

# Only needed if the base image lacks these -- see kaggle_build_wheels.ipynb.
# No internet here: this notebook runs offline during the official scored rerun,
# so a missing wheel with no wheels Dataset attached is a hard failure, not a
# fallback-to-internet situation.
WHEELS_DATASET = Path('/kaggle/input/cell-tracking-wheels')  # adjust to the attached Dataset's mount name

for pkg in ('blosc2', 'zarr', 'zstandard'):
    try:
        __import__(pkg)
    except ImportError:
        assert WHEELS_DATASET.exists(), (
            f'{pkg} is not preinstalled and no wheels Dataset is attached at '
            f'{WHEELS_DATASET}. Run notebooks/kaggle_build_wheels.ipynb once '
            '(internet on) and attach its output Dataset here.'
        )
        # Kaggle preserves /kaggle/working's directory structure when it becomes
        # an Output Dataset, so the .whl files may be nested under a subfolder
        # rather than sitting at the Dataset root -- search instead of assuming.
        wheel_dirs = sorted({p.parent for p in WHEELS_DATASET.rglob('*.whl')})
        assert wheel_dirs, f'no .whl files found anywhere under {WHEELS_DATASET}'
        find_links = []
        for d in wheel_dirs:
            find_links += ['--find-links', str(d)]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                        *find_links, pkg], check=True)


In [ ]:
import json, os, subprocess, sys
from pathlib import Path

SRC_DATASET = Path('/kaggle/input/cell-tracking-src')  # adjust to the attached Dataset's mount name
assert SRC_DATASET.exists(), f'code Dataset not attached at {SRC_DATASET}'
sys.path.insert(0, str(SRC_DATASET / 'src'))
# subprocess.run below spawns a FRESH Python process that does not
# inherit this sys.path.insert -- it needs PYTHONPATH in its own env.
SRC_ENV = {**os.environ, 'PYTHONPATH': str(SRC_DATASET / 'src')}

manifest = json.loads((SRC_DATASET / 'ARTIFACT_MANIFEST.json').read_text())
print('artifact manifest:', json.dumps(manifest, indent=1))
assert manifest.get('checkpoint'), 'no checkpoint recorded in this artifact -- inference will refuse to run'

In [ ]:
from cell_tracking import config
print('on_kaggle:', config.on_kaggle())
print('test_dir:', config.get_test_dir())
print('resolved checkpoint:', config.get_checkpoint())
assert config.get_checkpoint() is not None, 'no detector.pt found -- check the attached Dataset'

In [ ]:
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'predict.py'),
                '--out-dir', '/kaggle/working/preds'], check=True, env=SRC_ENV)

In [ ]:
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'make_submission.py'),
                '--geff-dir', '/kaggle/working/preds', '--out', 'submission.csv'], check=True, env=SRC_ENV)